# Blue Core De-Duplicate Agent

In [1]:
import asyncio
import os
import sys
import uuid

import nest_asyncio
import rdflib

from bluecore_models.utils.graph import init_graph, BF
from pydantic_ai.models.openai import OpenAIChatModel
from pydantic_ai.providers.openai import OpenAIProvider
from pymilvus import model, MilvusClient

sys.path.append("../bluecore_agents/src/")
from bluecore_ai.agents.duplicate import SupportDependencies, agent as dedup_agent

python-dotenv could not parse statement starting at line 10
None of PyTorch, TensorFlow >= 2.0, or Flax have been found. Models won't be available and only tokenizers, configuration and file/data utilities can be used.


In [2]:
milvus = MilvusClient(url="http://localhost:19530")
stanford_model = OpenAIChatModel(
    'gpt-4o',
    provider=OpenAIProvider(
        base_url=os.environ['PROVIDER_URL'], 
        api_key=os.environ['PROVIDER_API_KEY']
    ),
)

dedup_agent.model = stanford_model
nest_asyncio.apply()

embedding_func = model.DefaultEmbeddingFunction()

## Test Known Title

In [3]:
work_graph_01 = init_graph()
work_graph_01_url = rdflib.URIRef(f"http://localhost:8000/works/{uuid.uuid4()}")
work_graph_01.add((work_graph_01_url, rdflib.RDF.type, BF.Work))
work_title_01 = rdflib.BNode()
work_graph_01.add((work_title_01, rdflib.RDF.type, BF.Title))
work_graph_01.add((work_graph_01_url, BF.title, work_title_01))
work_graph_01.add((work_title_01, BF.mainTitle, rdflib.Literal("Samayaśāstra śāstrānusāriṇī ṭīkā")))
work_title_02 = rdflib.BNode()
work_graph_01.add((work_title_02, rdflib.RDF.type, BF.VariantTitle))
work_graph_01.add((work_graph_01_url, BF.title, work_title_02))
work_graph_01.add((work_title_02, BF.mainTitle, rdflib.Literal("Samayasar of Acharya Kundakunda with shastranusarini tika")))

<Graph identifier=N4381855cf95f4b41984eb525a094c959 (<class 'rdflib.graph.Graph'>)>

In [4]:
print(work_graph_01.serialize(format='turtle'))

@prefix bf: <http://id.loc.gov/ontologies/bibframe/> .

<http://localhost:8000/works/ed6e152c-2919-4b28-8dcc-2f417b58c8e4> a bf:Work ;
    bf:title [ a bf:Title ;
            bf:mainTitle "Samayaśāstra śāstrānusāriṇī ṭīkā" ],
        [ a bf:VariantTitle ;
            bf:mainTitle "Samayasar of Acharya Kundakunda with shastranusarini tika" ] .




In [5]:
result = dedup_agent.run_sync("Please evaluate this rdf",
                              deps=SupportDependencies(
                                  milvus_client=milvus,
                                  embedding_func=embedding_func,
                                  incoming_rdf=work_graph_01.serialize(format='json-ld')))

In [6]:
result

AgentRunResult(output=DeDupResult(score=0.9844595193862915, best_match='http://localhost:8000/works/2321d1ef-3e34-438e-9bc6-250fe2733ca4'))